# MuonClip angular/radial RG diagnostics: initial / best / final

This exploratory notebook uses the **same shared three-checkpoint implementation** as the canonical angular notebook. It loads actual saved `initial`, `best`, and `final` checkpoints, analyzes all six matrices, and then filters the displayed table to `RG_MATRIX_NAME`.

The angular power-law fit always uses `powerlaw.Fit(all_positive_values, discrete=False, verbose=False)` with **no manual `xmin` and no `xmax`**. The package chooses the tail start and retains the largest observed values.

The three angular flows are:

\[
initial\to best,\qquad initial\to final,\qquad best\to final.
\]

Each is compared with its own matched random-angular Haar/Stiefel null.

Papermill uses the same environment variables as the canonical notebook:

```bash
export RG_OPTIMIZERS_ROOT=/path/to/rg_optimizers
export RUNROOT=/tmp/<training-run-root>
export RESULTS_ROOT="$RUNROOT/results"
export TARGET_OPTIMIZER=muon_clip
export TARGET_SEED=4242
export RUN_DIR="$RESULTS_ROOT/$TARGET_OPTIMIZER/seed_$TARGET_SEED"
export RG_MATRIX_NAME=L00_W_Q
export ANGULAR_N_NULL=500

papermill \
  baseline/nanogpt_one_head/notebooks/muonclip_angular_radial_rg.ipynb \
  /tmp/muonclip_angular_rg_seed${TARGET_SEED}.out.ipynb
```

Optional overrides are `INITIAL_CHECKPOINT_PATH`, `BEST_CHECKPOINT_PATH`, `FINAL_CHECKPOINT_PATH`, and `ANGULAR_OUTPUT_DIR`.


In [ ]:
from pathlib import Path
import os
import sys

def find_experiment_root() -> Path:
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Set RG_OPTIMIZERS_ROOT or launch from the rg_optimizers repository"
    )

EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
print("EXPERIMENT_ROOT =", EXPERIMENT_ROOT)


In [ ]:
from IPython.display import display
from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
from rg_nanogpt_one_head.angular_three_checkpoint import run_three_checkpoint_analysis

CONFIG = AnalysisConfig.from_env()
MATRIX_NAME = os.environ.get("RG_MATRIX_NAME", "L00_W_Q")
print(CONFIG)
print("RG_MATRIX_NAME =", MATRIX_NAME)


In [ ]:
RESULTS, MANIFEST = run_three_checkpoint_analysis(CONFIG)
selected = RESULTS[RESULTS["matrix_name"] == MATRIX_NAME]

print("Checkpoints:")
for state, path in MANIFEST["checkpoints"].items():
    print(f"  {state:7s} step={MANIFEST['steps'][state]:7d}  {path}")

display(selected if len(selected) else RESULTS)

print("\nInterpret pairwise flow using actual alpha/xmin/tail length against the random-null interval.")


## Interpretation

Use `initial->best` to ask whether angular organization appears by the best-validation state, `initial->final` for the total learned angular flow, and `best->final` to see whether later training continues to organize the angular sector or instead drifts back toward the random baseline.

For the selected matrix, inspect both `tilt` and `twist`; for square attention matrices the twist sector is generally the informative one.

The pairwise alpha summary plot and far-tail CCDF plots are the primary diagnostics. They compare the fitted exponent and long-tail structure directly against matched random-angular null realizations.
